# INVEN 질문&답변 게시판 DATA ANALYSIS

### **<font color=yellow>뉴비/복귀 유저 관련 키워드 포함</font>**

#### 1. RAW DATA
- csv 위치 : data / processed
- csv 파일명 : inven_question_final.csv
- 노트북 위치 : ./notebooks

#### 2. 라이브러리 추가
- kiwipiepy
- wordcloud

#### 3. 데이터 전처리 후 csv 요약
<img src='../images/data_structure_eda.webp' height=450>

In [1]:
import re
import json
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from kiwipiepy import Kiwi
from wordcloud import WordCloud
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from collections import Counter, defaultdict    # 카테고리별로 Counter를 각각 가질 수 있는 딕셔너리 생성

import platform

sns.set_theme(style='whitegrid')

# 한글 폰트 — 실행 중인 OS 에 맞춰 자동 설정
if platform.system() == 'Windows':
    KOREAN_FONT = 'Malgun Gothic'
    FONT_PATH = 'C:/Windows/Fonts/malgun.ttf'
elif platform.system() == 'Darwin':          # macOS
    KOREAN_FONT = 'AppleGothic'
    FONT_PATH = '/System/Library/Fonts/Supplemental/AppleGothic.ttf'
else:                                        # Linux (Colab 등)
    KOREAN_FONT = 'NanumGothic'
    FONT_PATH = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'

plt.rcParams['font.family'] = KOREAN_FONT
plt.rcParams['axes.unicode_minus'] = False   # 마이너스(−) 부호 깨짐 방지

kiwi = Kiwi()   # 한국어 형태소 분석기(자바 불필요)


In [2]:
# RAW DATA 호출
qna_csv = pd.read_csv('../data/processed/inven_question_final.csv')
#qan_df = pd.DataFrame()

print('DATA SIZE :', qna_csv.shape)
print('\nCATEGORY')
print(f'카테고리 수: {len(qna_csv['category'].unique())}개 [{qna_csv['category'].unique()}]')
print('\nDATA INFO')
print(qna_csv.info())
print('\nDATA HEAD')
display(qna_csv.head(3))


DATA SIZE : (4157, 12)

CATEGORY
카테고리 수: 6개 [['기타' '아이템' '퀘스트' '직업' '시세' '몬스터']]

DATA INFO
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4157 entries, 0 to 4156
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   category             4157 non-null   object
 1   title                4157 non-null   object
 2   created_at           4157 non-null   object
 3   views                4157 non-null   int64 
 4   likes                4157 non-null   int64 
 5   content              4157 non-null   object
 6   comment_count        4157 non-null   int64 
 7   title_clean          4157 non-null   object
 8   content_clean        4156 non-null   object
 9   analysis_text        4157 non-null   object
 10  has_question_signal  4157 non-null   bool  
 11  is_question          4157 non-null   bool  
dtypes: bool(2), int64(3), object(7)
memory usage: 333.0+ KB
None

DATA HEAD


,category,title,created_at,views,likes,content,comment_count,title_clean,content_clean,analysis_text,has_question_signal,is_question
0,기타,모멘텀패스 살까요 플러스 나오는 거 살까요?,2026-08-19,36,0,챌섭에서 렌 키우고 있습니다.\r\n지금 렙 284인데 모멘텀패스 사고 다음 주 에...,1,모멘텀패스 살까요 플러스 나오는 거 살까요?,챌섭에서 렌 키우고 있습니다. 지금 렙 284인데 모멘텀패스 사고 다음 주 에픽던전...,모멘텀패스 살까요 플러스 나오는 거 살까요? 챌섭에서 렌 키우고 있습니다. 지금 렙...,True,True
1,기타,자석펫 확률업 공지 하고 하나요??,2026-08-19,143,0,자석펫 사야 하는데\r\n혹시 내일 패치하면 들어올까 해서\r\n공지 하고 하나요?...,0,자석펫 확률업 공지 하고 하나요??,자석펫 사야 하는데 혹시 내일 패치하면 들어올까 해서 공지 하고 하나요?? 공지 안...,자석펫 확률업 공지 하고 하나요?? 자석펫 사야 하는데 혹시 내일 패치하면 들어올까...,True,True
2,아이템,울티마 상점떔에 3배 썩어나는데 4배로 교환할수있나요??,2026-08-19,157,0,받기전 3배 이벤창에서밖에 교환못함요...?,2,울티마 상점떔에 3배 썩어나는데 4배로 교환할수있나요??,받기전 3배 이벤창에서밖에 교환못함요..?,울티마 상점떔에 3배 썩어나는데 4배로 교환할수있나요?? 받기전 3배 이벤창에서밖에...,True,True


### TOKENIZE


In [44]:
# 질문/답변 게시판의 단어 Tokenize

with open('../data/processed/stopwords_ko.json', encoding='utf-8') as f:
    stopwords_general = set(json.load(f))

# 뉴비/복귀 유저 관련 데이터를 추출하기 위한 수정된 불용어
stopwords_domain = {'정도', '질문', '드리', '키우', '이번', '맞추', '고민', '생각', '스펙', 
'나오', '어떻', '돌리', '모르', '부탁', '가능', '목표', '올리', '구매', '메이플', '바꾸', '상태', '안녕', 
'사용', '바르', '시간'}

stopwords_all = stopwords_general | stopwords_domain

findwords_domain = {
    "뉴비",
    "메린이",
    "메린",
    "복귀",
    "복귀유저",
    "유입",
    "입문",
    "초보",
}

def tokenize(text):
    """정제 → 형태소 → 품사 필터 → 2글자 이상 → 불용어 제거 → 중복 제거(순서 보존)."""
    tokens = [
        t.form for t in kiwi.tokenize(text)
        if t.tag.startswith(('NN', 'VA', 'VV'))          # 명사·형용사·동사만
        and len(t.form) > 1 
        and t.form not in stopwords_all
    ]
    return list(dict.fromkeys(tokens))  # 중복 제거 (순서 유효)

In [45]:
# 원본 csv 복제
qna_anal = qna_csv.copy()

# 질문/답변 게시판 데이터를 모두 토큰으로 바꾼다.
tokens_list = [tokenize(t) for t in qna_anal['analysis_text']]
qna_anal['tokens'] = tokens_list

display(qna_anal.info())
display(qna_anal.head(3))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4157 entries, 0 to 4156
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   category             4157 non-null   object
 1   title                4157 non-null   object
 2   created_at           4157 non-null   object
 3   views                4157 non-null   int64 
 4   likes                4157 non-null   int64 
 5   content              4157 non-null   object
 6   comment_count        4157 non-null   int64 
 7   title_clean          4157 non-null   object
 8   content_clean        4156 non-null   object
 9   analysis_text        4157 non-null   object
 10  has_question_signal  4157 non-null   bool  
 11  is_question          4157 non-null   bool  
 12  tokens               4157 non-null   object
dtypes: bool(2), int64(3), object(8)
memory usage: 365.5+ KB


None

,category,title,created_at,views,likes,content,comment_count,title_clean,content_clean,analysis_text,has_question_signal,is_question,tokens
0,기타,모멘텀패스 살까요 플러스 나오는 거 살까요?,2026-08-19,36,0,챌섭에서 렌 키우고 있습니다.\r\n지금 렙 284인데 모멘텀패스 사고 다음 주 에...,1,모멘텀패스 살까요 플러스 나오는 거 살까요?,챌섭에서 렌 키우고 있습니다. 지금 렙 284인데 모멘텀패스 사고 다음 주 에픽던전...,모멘텀패스 살까요 플러스 나오는 거 살까요? 챌섭에서 렌 키우고 있습니다. 지금 렙...,True,True,"[모멘텀, 패스, 플러스, 챌섭, 사고, 에픽던전, 익몬, 내일, 프리미엄, 하메,..."
1,기타,자석펫 확률업 공지 하고 하나요??,2026-08-19,143,0,자석펫 사야 하는데\r\n혹시 내일 패치하면 들어올까 해서\r\n공지 하고 하나요?...,0,자석펫 확률업 공지 하고 하나요??,자석펫 사야 하는데 혹시 내일 패치하면 들어올까 해서 공지 하고 하나요?? 공지 안...,자석펫 확률업 공지 하고 하나요?? 자석펫 사야 하는데 혹시 내일 패치하면 들어올까...,True,True,"[자석펫, 확률업, 공지, 자석, 내일, 패치, 들어오]"
2,아이템,울티마 상점떔에 3배 썩어나는데 4배로 교환할수있나요??,2026-08-19,157,0,받기전 3배 이벤창에서밖에 교환못함요...?,2,울티마 상점떔에 3배 썩어나는데 4배로 교환할수있나요??,받기전 3배 이벤창에서밖에 교환못함요..?,울티마 상점떔에 3배 썩어나는데 4배로 교환할수있나요?? 받기전 3배 이벤창에서밖에...,True,True,"[울티마, 상점, 교환]"


### 단어 빈도

질문/답변 게시판에서 가장 많이 본 글의 핵심 단어 확인

In [46]:
# 가장 많이 본 글의 핵심 단어를 센다
pos_counter = Counter()

for views, tokens in zip(qna_anal['views'], qna_anal['tokens']):
    if views >= 1:
        pos_counter.update(tokens)

compare = pd.DataFrame({
    '가장 많이 본 글의 단어 top20': [f'{w} ({n})' for w, n in pos_counter.most_common(20)],
}, index=range(1, 21))
display(compare)

,가장 많이 본 글의 단어 top20
1,챌섭 (1028)
2,뉴비 (500)
3,무기 (459)
4,해방 (416)
5,보조 (415)
6,보스 (381)
7,메린 (366)
8,직업 (304)
9,레테 (303)
10,미트라 (292)


### 카테고리별 키워드 상위 5개

질문/답변 게시판 카테고리별 가장 많이 본 글의 핵심 단어 확인

In [47]:
# 카테고리별 핵심 키워드 5개 확인

# 카테고리별로 Counter를 각각 가질 수 있는 딕셔너리 생성
from collections import Counter, defaultdict

pos_counter_by_cat = defaultdict(Counter)

# 1. 카테고리별로 조건에 맞는 tokens 집계
for category, views, tokens in zip(qna_anal['category'], qna_anal['views'], qna_anal['tokens']):
    if views >= 1:
        # 해당 카테고리의 Counter에만 단어 개수 누적
        pos_counter_by_cat[category].update(tokens)

# 2. 카테고리별로 상위 5개 단어 추출 및 출력
for cat, counter in pos_counter_by_cat.items():
    # counter.most_common(5)로 상위 5개 (단어, 개수) 추출
    top5_words = [f'{w} ({n})' for w, n in counter.most_common(5)]
    
    compare = pd.DataFrame({
        f'"{cat}" 카테고리에서 가장 많이 언급된 단어 top5': top5_words
    }, index=range(1, 6))
    
    display(compare)

,"""기타"" 카테고리에서 가장 많이 언급된 단어 top5"
1,챌섭 (350)
2,보스 (139)
3,뉴비 (135)
4,패스 (128)
5,사냥 (117)


,"""아이템"" 카테고리에서 가장 많이 언급된 단어 top5"
1,챌섭 (505)
2,무기 (386)
3,보조 (352)
4,미트라 (260)
5,뉴비 (255)


,"""퀘스트"" 카테고리에서 가장 많이 언급된 단어 top5"
1,해방 (31)
2,보스 (21)
3,퀘스트 (18)
4,챌섭 (17)
5,뉴비 (14)


,"""직업"" 카테고리에서 가장 많이 언급된 단어 top5"
1,직업 (122)
2,챌섭 (113)
3,추천 (90)
4,레테 (77)
5,뉴비 (66)


,"""시세"" 카테고리에서 가장 많이 언급된 단어 top5"
1,시세 (53)
2,가격 (37)
3,챌섭 (31)
4,팔리 (17)
5,뉴비 (16)


,"""몬스터"" 카테고리에서 가장 많이 언급된 단어 top5"
1,배율 (40)
2,패턴 (22)
3,보스 (22)
4,극딜 (20)
5,하드 (19)


### 카테고리별 VIEWS 합계

In [48]:
# 카테고리별 VIEWS 합계 확인
print(f'카테고리별 VIEWS\n\n{qna_anal.groupby('category')['views'].sum()}')

카테고리별 VIEWS

category
기타     1444263
몬스터     143017
시세      147318
아이템    1827157
직업      342270
퀘스트     127602
Name: views, dtype: int64


### 카테고리별 질문 수

In [49]:
# 카테고리별 질문 수 확인
display(qna_anal.groupby('category')['analysis_text'].agg(['count']))

,count
category,
기타,1332
몬스터,144
시세,156
아이템,2031
직업,385
퀘스트,109


### VIEWS 상위 게시글 5개

In [50]:
# VIEWS 상위 게시글 5개 확인
display(qna_anal.sort_values('views', ascending=False)[['category', 'views', 'analysis_text', 'tokens']].head())

,category,views,analysis_text,tokens
3960,기타,35501,노말 카이 최소컷이 몇이에요? 지금 레테 리레4랩 컨티3랩있고 투력 1300만인데 ...,"[카이, 최소, 레테, 컨티, 투력]"
858,기타,31777,"울티마 3-10 깨려면 법사도 40 찍어야대요? 전사 40, 궁수 40 찍고 스킬배...","[울티마, 법사, 전사, 궁수, 스킬, 배우, 밀리]"
3382,기타,30880,레테 어빌리티 질문요 레테 어빌리티 돌리다가 2번째줄에 상추뎀8%가 떳는데 자물쇠 ...,"[레테, 어빌리티, 상추뎀, 자물쇠, 잠구, 서큘레이터]"
2101,아이템,22317,챌린저목표 8600만 윈브 뭘 더 해야할까요? 제목 그대로 챌린저목표로 달리고 있습...,"[챌린저, 윈브, 제목, 달리, 닉네임, 윈브프라임, 무기, 해방, 미트, 보조, ..."
3600,기타,18753,이번 챌섭 페어리하트작 뭐가 맞아요..? 다 말이 조금씩 달라서 대충 여론이 1. ...,"[챌섭, 페어리, 하트, 다르, 여론, 피버, 본섭, 넘어오, 매지컬, 상점, 코인..."


### 카테고리별 VIEWS 상위 10개 게시글

In [51]:
# 카테고리별 'views' 기준 상위 10개 게시글 확인
top_n_posts = (
    qna_anal.sort_values(by=['category', 'views'], ascending=[True, False])
    .groupby('category')
    .head(10)
)
for category, group in top_n_posts.groupby('category'):
    print(f"[{category}] 카테고리 상위 10개 게시글")
    display(group[['title', 'views', 'content', 'tokens']].reset_index(drop=True))
    print("\n" + "=" * 100 + "\n")


[기타] 카테고리 상위 10개 게시글


,title,views,content,tokens
0,노말 카이 최소컷이 몇이에요?,35501,지금 레테 리레4랩 컨티3랩있고 투력 1300만인데 갈수있나요?,"[카이, 최소, 레테, 컨티, 투력]"
1,울티마 3-10 깨려면 법사도 40 찍어야대요?,31777,"전사 40, 궁수 40 찍고 스킬배워서 밀 수 있을 줄 알았는데 택도 없네요.\r\...","[울티마, 법사, 전사, 궁수, 스킬, 배우, 밀리]"
2,레테 어빌리티 질문요,30880,레테 어빌리티 돌리다가 2번째줄에 상추뎀8%가 떳는데\r\n자물쇠 잠궈도 되나요? ...,"[레테, 어빌리티, 상추뎀, 자물쇠, 잠구, 서큘레이터]"
3,이번 챌섭 페어리하트작 뭐가 맞아요..?,18753,다 말이 조금씩 달라서\r\n대충 여론이\r\n1. 일단 피버때 30퍼 작하고 본섭...,"[챌섭, 페어리, 하트, 다르, 여론, 피버, 본섭, 넘어오, 매지컬, 상점, 코인..."
4,챌 1섭을 해야하는 이유,17838,가 있나요??.. 3까지는 다 가능한거 아닌지요...,"[1섭, 이유]"
5,챌섭에서 이번에 처음 키우려고 하는데 200렙 달성 비약 질문드려요,15301,본섭에서는 유니온 9240인데 주보용 부캐 하나 키우고싶어서 챌섭에서 키우고 본섭으...,"[챌섭, 처음, 200렙, 비약, 본섭, 유니온, 주보, 부캐, 리프, 상황, 예전..."
6,스인미 크오솔 쓰는 이유?,13485,저 잘 몰라서 그런데 다들 스인미 크오솔 쓰시는이유가 뭐에요 딜도 약하고 효과읽어보...,"[스인미, 크오솔, 이유, 약하, 효과, 증가]"
7,챌섭 무자본 현실적 목표,11288,챌섭 오늘부터하면 티어 어디로 목표 잡아야할까요? 챌린저패스만 샀고 현질은 더 안할...,"[챌섭, 자본, 현실, 오늘, 챌린저, 패스, 현질]"
8,연합 토큰 어떻게 얻나요?,11013,미션 울티마인지 뭔 씨잘떼기 없는 스토리텔링 영상 따위에 무슨 시간 할애를\r\n1...,"[연합, 토큰, 미션, 울티마, 씨잘떼기, 스토리텔링, 영상, 할애, 강제, 지랄,..."
9,지피방? 원격피시방? 질문있습니다,10928,안녕하세요. 챌섭 유입뉴비입니다\r\n이번달은 간신히 피시방 15시간 채워서 자석펫...,"[지피방, 원격, 피시, 챌섭, 유입, 뉴비, 채우, 자석, 직장, 다니, 피시방,..."




[몬스터] 카테고리 상위 10개 게시글


,title,views,content,tokens
0,메이린 확률,6157,세삼스레 궁금한건데 노말 메이린 칠흑상자 드랍률이랑 조각상자 확률 몇이지??\r\n...,"[메이, 확률, 칠흑상자, 드랍률, 조각상자, 하급]"
1,원래 세렌 정신병자가만든 보스맞나요?,5101,제가 트라이하다가 정신병걸릴거같은데 하드듄캘이랑 두마리 개초딩패턴으로 돈뽑아처먹을려...,"[원래, 세렌, 정신병자, 만들, 보스, 트라이, 정신병, 걸리, 하드듄캘, 마리,..."
2,메이린 계속 5퍼정도 남는데 어캐 하면 잡을까요,4062,https://youtu.be/uBWIXsQVTnw\r\n4.6보마 96퍼센트 나옵...,"[메이, 어캐, 퍼센트, 애초, 개똥, 보우마스터, 직업, 용기, 시도, 묶이, 건..."
3,하드 메이린이랑 노말 메이린 차이가 큰가요?,3718,노말 메이린 114퍼로 깨서 이왕 할거 챌린저 달아보려고 하는데요!!\r\n템펙업을...,"[하드, 메이, 노말, 차이, 챌린저, 템펙업, 하드메이린, 배율]"
4,검마 아케인포스 충족한데 반감이에요,3336,현재 아케인포스 1420인데 반감 받고 있어요\r\n아케인포스 어디까지 올려야할까요?,"[검마, 아케인포스, 충족, 반감]"
5,챌섭 일일보스 필수인가요?,2896,길드 가입은 했는데 기보 가는 사람들이 없어요\r\n혼자서 일일보스 매일 도는게 좋...,"[챌섭, 일일보스, 필수, 길드, 가입, 사람, 처음, 자본, 무과금, 유저]"
6,진짜 급해서 그러는데 흉성 2정수빌드 설명 좀,2806,항상 환상>현실로 넘어가는 첫 극딜 때 최종데미지가 130이 아니라 120일 때 극...,"[급하, 그러, 흉성, 정수빌드, 설명, 환상, 현실, 넘어가, 최종, 데미지, 극..."
7,검마 쩔받을때 받는 사람 배율은 왜 필요하나요?,2518,왜죵??\r\n배율은 113퍼 나오고 패턴은 모릅니다\r\n보통 가격이랑 상자 같이...,"[사람, 배율, 필요, 패턴, 가격, 상자]"
8,포뻥 vs 그냥 검마 잡기,2401,메린이 검마 솔격 트라이 박으려는데 그냥 하면 124퍼 정도임\r\n근데 하이퍼스탯...,"[포뻥, 메린, 트라이, 하이퍼스탯, 보약, 피방칭호, 합치, 보뎀, 뎀3퍼, 방무..."
9,이지카링 많이 어렵나요?,2001,하드세렌 잡고 이제 이지카링 차례인데 많이 어렵나요? 배율은 138퍼 나오고 하드세...,"[이지카링, 어렵, 하드세렌, 차례, 배율, 클리어]"




[시세] 카테고리 상위 10개 게시글


...

,title,views,content,tokens
0,챌린저목표 8600만 윈브 뭘 더 해야할까요?,22317,제목 그대로 챌린저목표로 달리고 있습니다.\r\n닉네임은 윈브프라임 입니다.\r\n...,"[챌린저, 윈브, 제목, 달리, 닉네임, 윈브프라임, 무기, 해방, 미트, 보조, ..."
1,챌섭 레잠 어디에써야되나요?¿?¿?¿?¿?¿?,13384,1. 블랙보조\r\n2. 엠블렘(미트라)\r\n3. 페어리하트\r\n당신의 선택은?...,"[챌섭, 레잠, 블랙, 보조, 엠블렘, 미트라, 페어리, 하트, 선택, 무기]"
2,제네무기 에디 레전,12483,그냥 에디 유닉2줄정도로 쓰면서 무료 화에큡 던져주고 돈모았다가 미라클때 쌍레가는게...,"[무기, 에디, 유닉, 무료, 화에큡, 던지, 모으, 미라클]"
3,도전자장비 11강 몇메소 정도임??,9533,챌린저 도전하는 메린이인데 챌섭끝나면 장비 맞춰야 하잖슴... 그럼 도전자장비 11...,"[도전자, 장비, 몇메소, 챌린저, 도전, 메린, 챌섭]"
4,에디는 등업 확률이 더 낮나요??,9144,에디 레어에서 에픽 가는데 브론즈큐브 한 300~400개 넣었는데 하나도 등업이 안...,"[에디, 확률, 레어, 에픽, 브론즈, 큐브, 잠재, 다르]"
5,전투력 1억이고 검마 솔격하고 접을 생각입니다,8928,지금 30억정도있고 이걸로 리레 4렙 살건데\r\n지금 가장 가성비 스펙업 알려주십쇼,"[전투력, 격하, 리레, 4렙, 가성비, 알리]"
6,미트라가 압도적으로 지지받는 이유가 뭔지 알 수 있을까요?,8709,옵션만 놓고보면 공마3에 주스텟30정도던데\r\n가격 차이는 너무 압도적으로 많이 ...,"[미트라, 압도, 지지, 이유, 옵션, 공마3, 주스텟, 가격, 차이, 일반, 엠블..."
7,챌섭 메린이 파풀마 대신 블빈마를 사버렸습니다..망한건가요?,8643,어떤 유튜버(지@)를 보고 파풀마가 있는줄 모르고 블빈마를 사버렸습니다..\r\n근...,"[챌섭, 메린, 파풀마, 대신, 블빈마, 망하, 유튜버, 메소, 초반, 거래, 가격..."
8,카레잠 사용처좀 추천해주세요.,8108,안녕하세요 선배님들.\r\n이전에 질문글 남기고 현재 챌린저스4에서 렌을 키우고있습...,"[카레, 사용처, 추천, 선배, 이전, 남기, 챌린저스, 제네패스, 무기, 생기, 답변]"
9,모멘텀 패스 구매고민,7357,안녕하세요\r\n오늘 나오는 모멘텀 패스를 누구 줄지 고민이라서 질문드립니다.\r\...,"[모멘텀, 패스, 템상황, 챌섭, 보우마스터, 활잡이육개장, 본섭, 묘소협, 누더기..."




[직업] 카테고리 상위 10개 게시글


,title,views,content,tokens
0,신규 직업 버프라던데,13541,"이번에 시작해보려고하는데, 원래는 레테가 생각보다 별로라는 평이 있어서 렌이랑 아란...","[신규, 직업, 버프, 시작, 원래, 레테, 아란, 원하, 성능캐릭, 그러, 처음,..."
1,저 지금 방금 막 헥사 떡작했는데..,8779,주스텟\r\n마력\r\n보뎀\r\n이렇게 맞추고 돌렸는데\r\n5510이 붙었는데....,"[헥사, 떡작, 주스텟, 마력, 보뎀, 초기]"
2,직업별 챌린저스 서버 시드링,5883,컨4리3 / 컨3리4 뭐골라야 하는지 어디 모아서 정리해둔 것 없을까요\r\n물론 ...,"[직업, 챌린저스, 서버, 시드, 컨4리, 고르, 모으, 정리, 한쪽, 주변, 처음..."
3,개초보뉴비 챌린저하고싶습니다 길을알려주세요,5471,지금 길라잡이 보스컨텐츠 순서 기준\r\n실제로는 노말루시드 / 이지 윌 잡았고\r...,"[초보, 뉴비, 챌린저, 알리, 길라잡이, 보스, 컨텐츠, 순서, 기준, 노말루시드..."
4,레테 헥사 스텟,3404,제가 헥사 스텟에 대해 잘 몰라서 그런데\r\n헥사 스텟 능력치 뭐하면 되나요?,"[레테, 헥사, 스텟, 제가, 대하, 능력]"
5,렌 챌섭 쿨뚝써야함?,3034,도전자셋 쿨뚝받아야함 스탯뚝 받아야함?,"[챌섭, 도전자, 스탯]"
6,도와주세요 전투력 2100만까지 올렸는데도,2945,아델 직업인데 전투력 2187되나? 그정도 되는데도 하드스우도 못깨고 하드데미안도 ...,"[전투력, 아델, 직업, 하드스우, 하드데미안, 노말듄켈, 힘들, 패턴, 피하, 어..."
7,솔헤카테는 어떻게 쓰면되나요?,2718,직업은 레테고\r\n헥사 강화 순서 보니까 마코 몇개 열고 헤카테를 열라길래 일단 ...,"[헤카테, 직업, 레테고, 헥사, 강화, 순서, 마코, 팩텀, 스킬, 보스, 소환,..."
8,챌섭 제로 1일차 정리 및 질문,2505,지금까지 제가 챌섭 진행한거랑 질문할거 올립니다.\r\n제로 무기 4형에서 챌섭 코...,"[챌섭, 제로, 정리, 지금, 진행, 무기, 코인, 스타포스, 사서, 주문서, 연합..."
9,뉴비 레테 vs 렌 vs 보마,2400,안녕하세요\r\n아무것도 모르는 뉴비입니다 이제 메이플 깔고 캐릭 만들어야해요\r\...,"[뉴비, 레테, 보마, 아무것, 캐릭, 만들, 기본, 추천, 시작, 게시판, 공략,..."




[퀘스트] 카테고리 상위 10개 게시글


,title,views,content,tokens
0,에테리온 아티팩트 코어 활성화 뭐해요?,9063,사냥시 솔 에르다 흭득량 하면 되나용??\r\n서버는 챌린저스에여!,"[에테리온, 아티팩트, 코어, 활성, 사냥, 에르다, 흭득량, 서버, 챌린저스]"
1,제네패스 플러스,7830,1. 연모로 잡은것도 소급적용 되나요??\r\n2. 리워드는 순서대로 잡아야 처지인...,"[제네패스, 플러스, 연모, 소급, 적용, 리워드, 순서, 처지, 인정, 인팟, 해..."
2,울티마 3-1 뺑뺑이가 나을까요,5262,어제 2-10 이어서 3-1 뚫었고 3-2 못 뚫은 상태고\r\n32 31 20 정...,"[울티마, 뺑뺑이, 5렙, 장비, 궁수, 모자, 전사, 법사, 무기, 레벨, 파밍,..."
3,검마 해방퀘 주간보스 잡고 제네시스 연습모드 해도 인정되나요??,4625,주간보스로 실수로 다 잡아버린경우\r\n해방퀘때 스우나 데미안잡을때\r\n제네시스 ...,"[검마, 해방퀘, 주간보스, 제네시스, 연습, 모드, 인정, 실수, 경우, 해방, ..."
4,제네시스패스 너무 어려워요,4249,이번에 챌섭으로 유입된 뉴비인데 게임이 너무 어렵습니다.\r\n스우를 잡으라는데 잡...,"[제네시스패스, 어렵, 챌섭, 유입, 뉴비, 게임, 스우, 2페이즈, 공략, 영상,..."
5,에테리온 아티팩트 1주차 질문있습니다,3292,추천은 몬파랑 에픽던전으로 되어있는데 사냥시 솔 에르다 추가효과보다 효율이 좋을까요?,"[에테리온, 아티팩트, 1주차, 추천, 파랑, 에픽던전, 사냥, 에르다, 추가, 효..."
6,스펙터 블래스트 경치 수령 관련,3252,스펙터 블래스트 월드 당이 아닌 계정 혹은 명의당 1회만 경치 수령 가능한가요..?...,"[스펙터, 블래스트, 경치, 수령, 관련, 월드, 계정, 명의, 이벤트, 페이지]"
7,하드메이린.... 왜 안될까요,3200,배율도 어느정도 되는거 같은데 메린이라 손 문제일까요 에반이나 호영 카드가 잘 안뜨...,"[하드메이린, 메린, 문제, 에반, 호영, 카드, 미치, 도움]"
8,[제네시스 무기] 사자왕 반 레온의 흔적 퀘스트 질문,2705,봉인된 제네시스 무기 착용 후에 반레온 하드 잡는거 아닌가요? 잡아도 안 올라가네요 ㅠㅠ,"[사자, 레온, 흔적, 퀘스트, 봉인, 제네시스, 무기, 착용, 반레온, 하드, 올라가]"
9,울티마 2-6깨고 전사 스킬 뭐끼나요?,2449,디바이드 새로 배웠는데 어떻게 껴야함?\r\n파밍용,"[울티마, 전사, 스킬, 디바이드, 배우, 파밍용]"


#### <font color=yellow>뉴비/복귀 유저가 언급된 게시글 검색</font>

In [52]:
# findwords_domain 이 포함된 게시물 검색

qna_anal_new = qna_anal['tokens'].apply(lambda x: any(word in findwords_domain for word in x))
display(qna_anal[qna_anal_new].shape)
display(qna_anal[qna_anal_new].head())

(1098, 13)

,category,title,created_at,views,likes,content,comment_count,title_clean,content_clean,analysis_text,has_question_signal,is_question,tokens
3,아이템,뉴비 템세팅 질문.....,2026-08-19,166,0,템환 6.7 헥환 5.7 찍힙니다\r\n익스우 솔플정도 목표인데 앞으로 템세팅 어떻...,4,뉴비 템세팅 질문...,템환 6.7 헥환 5.7 찍힙니다 익스우 솔플정도 목표인데 앞으로 템세팅 어떻게 해...,뉴비 템세팅 질문... 템환 6.7 헥환 5.7 찍힙니다 익스우 솔플정도 목표인데 ...,True,True,"[뉴비, 세팅, 템환, 헥환, 찍히, 익스, 효율, 알리]"
10,기타,챌섭 지금 시점에서 뉴비가 검마 2~3인트라이하려면 어떤구인방법이 있을까요,2026-08-19,392,0,안녕하세요 감사합니다\r\n이번에 뉴비친구가 검마배율 100퍼정도 도달해서 2~3인...,1,챌섭 지금 시점에서 뉴비가 검마 2~3인트라이하려면 어떤구인방법이 있을까요,안녕하세요 감사합니다 이번에 뉴비친구가 검마배율 100퍼정도 도달해서 2~3인팟 꾸...,챌섭 지금 시점에서 뉴비가 검마 2~3인트라이하려면 어떤구인방법이 있을까요 안녕하세...,True,True,"[챌섭, 시점, 뉴비, 인트라이, 구인, 방법, 감사, 친구, 배율, 도달, 인팟,..."
11,아이템,샤타때 만들어버렸는데 이후 템셋 문의… 9-10만목표,2026-08-19,675,0,샤타때 고근 17->18을 한번만 눌러야지 했다가 바로터져서 홧김에 자동강화 돌렸는...,4,샤타때 만들어버렸는데 이후 템셋 문의… 9-10만목표,샤타때 고근 17->18을 한번만 눌러야지 했다가 바로터져서 홧김에 자동강화 돌렸는...,샤타때 만들어버렸는데 이후 템셋 문의… 9-10만목표 샤타때 고근 17->18을 한...,True,True,"[샤타, 만들, 이후, 문의, 고근, 누르, 터지, 홧김, 자동, 강화, 멈추, 벨..."
24,퀘스트,아버 미션 노말보스 하드로 깨도 되나요?,2026-08-18,265,0,메린이 투력 2500따리긴한데 진힐라 노멀 대신 하드로 해볼까합니다,2,아버 미션 노말보스 하드로 깨도 되나요?,메린이 투력 2500따리긴한데 진힐라 노멀 대신 하드로 해볼까합니다,아버 미션 노말보스 하드로 깨도 되나요? 메린이 투력 2500따리긴한데 진힐라 노멀...,True,True,"[아버, 미션, 보스, 하드, 메린, 투력, 따리, 진힐라, 노멀, 대신]"
43,기타,메린이 챌섭에서 사전리프 했다가 망했어요..,2026-08-18,3840,1,메이플랜드만 하다가 본 메이플 시작한지 일주일 정도 됐는데\r\n챌린지 서버에 처음...,25,메린이 챌섭에서 사전리프 했다가 망했어요..,메이플랜드만 하다가 본 메이플 시작한지 일주일 정도 됐는데 챌린지 서버에 처음으로 ...,메린이 챌섭에서 사전리프 했다가 망했어요.. 메이플랜드만 하다가 본 메이플 시작한지...,True,True,"[메린, 챌섭, 사전리프, 망하, 랜드, 시작, 일주일, 챌린지, 서버, 처음, 만..."


#### <font color=yellow>뉴비/복귀 유저가 언급된 카테고리별 VIEWS 상위 10개 게시글</font>

In [53]:
# 찾고자 하는 단어(findwords_domain)가 언급된 게시물 출력

# 1. tokens 리스트와 findwords_domain의 교집합이 존재하는지 확인 (isdisjoint 활용)
condition = qna_anal['tokens'].apply(
    lambda x: not findwords_domain.isdisjoint(x) if isinstance(x, (list, set)) else False
)

# 2. 필터링 및 카테고리별 상위 10개 추출
top_n_posts = (
    qna_anal[condition]
    .sort_values(by=['category', 'views'], ascending=[True, False])
    .groupby('category')
    .head(10)
)

for category, group in top_n_posts.groupby('category'):
    print(f"[{category}] - 단어 '{findwords_domain}' 포함 상위 10개 게시글")
    display(group[['title', 'views', 'content', 'tokens']].reset_index(drop=True))
    print("\n" + "=" * 100 + "\n")

[기타] - 단어 '{'뉴비', '초보', '메린이', '메린', '복귀', '복귀유저', '입문', '유입'}' 포함 상위 10개 게시글


,title,views,content,tokens
0,이번 챌섭 페어리하트작 뭐가 맞아요..?,18753,다 말이 조금씩 달라서\r\n대충 여론이\r\n1. 일단 피버때 30퍼 작하고 본섭...,"[챌섭, 페어리, 하트, 다르, 여론, 피버, 본섭, 넘어오, 매지컬, 상점, 코인..."
1,지피방? 원격피시방? 질문있습니다,10928,안녕하세요. 챌섭 유입뉴비입니다\r\n이번달은 간신히 피시방 15시간 채워서 자석펫...,"[지피방, 원격, 피시, 챌섭, 유입, 뉴비, 채우, 자석, 직장, 다니, 피시방,..."
2,뉴비 솔 에르다 / 솔 에르다 조각 선택권 받을라 하는데 [답답함주의],9440,사진에서 보는바와 같이 7월2일 까지 써라고 해서 사라질게 두려워서 미리 받아둘려고...,"[뉴비, 에르다, 조각, 선택, 주의, 사진, 사라지, 두렵, 지인, 솔에르다, 캐..."
3,헥사 코어 3 때문에 벽에 막힌거 같습니다,5570,이번 유입 메린이\r\n보마로 281렙 찍고 검밑솔 보스들 다 돌면서 행메중이었는데...,"[헥사, 코어, 때문, 막히, 유입, 메린, 보마, 보스, 행메, 부담, 무시, 에..."
4,메린이 이러면 5주 제네 해방 가능함??,4393,이번주 시간 없어서 그런데\r\n1주차 노말 검밑솔(진힐라x)\r\n2주차 검밑솔 ...,"[메린, 이러, 해방, 진힐라, 2주차, 검밑솔, 노말, 3주차, 검마, 주차]"
5,메린이 챌섭에서 사전리프 했다가 망했어요..,3840,메이플랜드만 하다가 본 메이플 시작한지 일주일 정도 됐는데\r\n챌린지 서버에 처음...,"[메린, 챌섭, 사전리프, 망하, 랜드, 시작, 일주일, 챌린지, 서버, 처음, 만..."
6,챌섭 9만점 목표로 할 때,3599,첫주차에 어디까지 보스 잡아야 된다 이런건 없죠??\r\n해방시기 당기고 싶을떄나 ...,"[챌섭, 보스, 해방, 시기, 당기, 뉴비, 개월]"
7,검마 2인격 뉴비기준,3545,검마 2인격 배율\r\n163.61% 나오는데 가능할까요 둘다 뉴비입니다..,"[인격, 뉴비, 기준, 배율]"
8,뉴비 이 캐릭터 버려야하는지 고민입니다..,2982,"273렙이고 저번챌섭때 키운거라 기본보장세트있습니다.\r\n무기,방어구 아무것도 없...","[뉴비, 캐릭터, 버리, 저번챌섭때, 기본, 보장, 세트, 무기, 방어, 엠블렘, ..."
9,데미지 스킨 얻는법,1972,챌섭 복귀한 불독 유저인데요\r\n기본 뎀스로 계속 하려니 좀 아쉬워서요..\r\n...,"[데미지, 스킨, 챌섭, 복귀, 불독, 유저, 기본, 뎀스, 아쉽, 익스플로전, 폭..."




[몬스터] - 단어 '{'뉴비', '초보', '메린이', '메린', '복귀', '복귀유저', '입문', '유입'}' 포함 상위 10개 게시글


,title,views,content,tokens
0,포뻥 vs 그냥 검마 잡기,2401,메린이 검마 솔격 트라이 박으려는데 그냥 하면 124퍼 정도임\r\n근데 하이퍼스탯...,"[포뻥, 메린, 트라이, 하이퍼스탯, 보약, 피방칭호, 합치, 보뎀, 뎀3퍼, 방무..."
1,검마 3페 질문,1481,썬콜 처음 검마 하는 뉴비인데요\r\n3페 극딜할때 버프쓰고 극딜 키다운 하면 끝에...,"[썬콜, 처음, 검마, 뉴비, 극딜할때, 버프쓰고, 극딜, 키다운, 버프, 타임, ..."
2,메린이 이적자 120퍼 가능한가요,1402,이칼 170퍼인데 1~2분 남기고 깨는 수준인데 이번주 초기화전까지 120퍼면 가능...,"[메린, 적자, 남기, 수준, 초기, 트라이]"
3,하드 데미안 1900만 보마 뉴비인데 ㅠㅠ 왜이렇게어렵나요,1385,꿀팁같은거 잇을까요... 연습모드라 풀도핑인데,"[하드, 데미안, 보마, 뉴비, 이렇, 어렵, 연습, 모드, 풀도핑]"
4,메린이 검마 솔격 가능함??,1104,하드 진 힐라 배율 210퍼 연모 3트 찐모 2트해서 15분쯤에 잡았는데\r\n검마...,"[하드, 힐라, 배율, 210퍼, 연모, 3트, 찐모, 2트, 내일, 하루, 종일,..."
5,검마 배율 181%인데 이거 잡히는거 맞나요..?,1097,방학이벤트마다 철새마냥 복귀하는 뉴비아닌 뉴비입니다....\r\n이번챌섭에서 레테 ...,"[잡히, 방학, 이벤트, 철새, 복귀, 뉴비, 이번챌섭, 레테, 해방, 주일, 앞당..."
6,메린이 드메템 사야되나요?,1081,하루 3~4시간 사냥하려는데 제미나이 피셜 드메템 복구하려면 40~50일이 걸린다고...,"[메린, 드메템, 하루, 사냥, 제미나이, 피셜, 복구, 걸리, 챌섭, 하드메이린,..."
7,검마 배율78퍼 2인 빡셀려나요?,1033,시즌3 유입이고 컨트롤 자신은 없는데 2인빡셀려나요?,"[배율, 빡세, 시즌, 유입, 컨트롤]"
8,메린이 하메 잡기까지 메소 얼마나 써야할까요?,871,썬콜하는 메린이인데 목표는 2~4주 뒤에 배율 105퍼이상을 찍는게 목표입니다.\r...,"[메린, 하메, 메소, 배율, 하루, 사냥, 최소, 안전, 찍히, 말씀, 감사]"
9,원래 보스 배율보다 훨씬 느리게 잡아지는게 정상임??,818,메린이 인데 보스 배율에 비해서 잡는시간이 너무 차이남;;\r\n카더랑 하윌 잡았는...,"[원래, 보스, 배율, 느리, 정상, 메린, 비하, 차이, 하위, 카더, 남기, 패..."




[시세] - 단어 '{'뉴비', '초보', '메린이', '메린', '복귀', '복귀유저', '입문', '유입'}' 포함 상위 10개 게시글


...

,title,views,content,tokens
0,도전자장비 11강 몇메소 정도임??,9533,챌린저 도전하는 메린이인데 챌섭끝나면 장비 맞춰야 하잖슴... 그럼 도전자장비 11...,"[도전자, 장비, 몇메소, 챌린저, 도전, 메린, 챌섭]"
1,챌섭 메린이 파풀마 대신 블빈마를 사버렸습니다..망한건가요?,8643,어떤 유튜버(지@)를 보고 파풀마가 있는줄 모르고 블빈마를 사버렸습니다..\r\n근...,"[챌섭, 메린, 파풀마, 대신, 블빈마, 망하, 유튜버, 메소, 초반, 거래, 가격..."
2,플라즈마 하트가 터져버렸어요..,5940,안녕하세요.. 챌3에서 280아란 키우고 있는 뉴비입니다..\r\n다름이 아니라 오...,"[플라즈마, 하트, 터지, 뉴비, 다르, 스타포스, 데브펜, 에스텔라, 확률, 멘붕..."
3,메소 다 파먹고도 재획 빨아도 이득인가영?,5533,"처음 메이플 챌섭에서 시작한 뉴비입니다\r\n다이아 티어에 아드, 메드는 어빌밖에 ...","[메소, 파먹, 이득, 처음, 챌섭, 시작, 뉴비, 다이아, 아드, 메드, 어빌]"
4,형님들 주에 30억버는 메린인데요,5318,템부터 사는게 좋을까요 아니면 조각이 우선일까요 하드메이린까진 가고싶습니다 조각은 ...,"[메린, 조각, 하드메이린]"
5,에테 18성 직작 vs 완작사기,5022,이번 챌섭으로 유입된 뉴비입니다.\r\n도전자 템 대체를 위해 상하의 에테 레전 2...,"[에테, 직작, 완작사기, 챌섭, 유입, 뉴비, 도전자, 대체, 위하, 상하, 준비..."
6,제네시스 해방무기 옵션 질문 드립니다.,4863,뉴비에 패스만 지른 유저 입니다.\r\n그 레잠 에유잠 발라서\r\n윗잠 보보공 4...,"[제네시스, 해방, 무기, 옵션, 뉴비, 패스, 지르, 유저, 레잠, 에유잠, 발라..."
7,여명셋 지금 많이 올랐나요? 템셋팅 알려주십쇼 ㅠㅠㅠ,4663,지금 여명 17성들이 가격이 많이 오른건가요?\r\n뉴비라 아무것도 모르네요\r\n...,"[여명, 오르, 셋팅, 알리, 가격, 뉴비, 효율, 평가, 부탁드리, 아래, 괜찮,..."
8,뉴비대박났는데 시세좀 알려주실 선생님ㅠ,4305,챌섭3이예요 미트라 화에큐돌리다 떴는데 가횟4인데 얼마에 팔릴까요? 적당한 시세 알...,"[뉴비, 시세, 알리, 선생, 챌섭, 미트라, 화에큐, 가횟4, 팔리, 감사]"
9,챌섭 뉴비 프펫공 프악공 질문,4273,프악공이랑 프펫공 사려는데 보스코인샵에서 사는게 정배인가요..?,"[챌섭, 뉴비, 프펫공, 악공, 보스, 코인]"




[직업] - 단어 '{'뉴비', '초보', '메린이', '메린', '복귀', '복귀유저', '입문', '유입'}' 포함 상위 10개 게시글


,title,views,content,tokens
0,신규 직업 버프라던데,13541,"이번에 시작해보려고하는데, 원래는 레테가 생각보다 별로라는 평이 있어서 렌이랑 아란...","[신규, 직업, 버프, 시작, 원래, 레테, 아란, 원하, 성능캐릭, 그러, 처음,..."
1,직업별 챌린저스 서버 시드링,5883,컨4리3 / 컨3리4 뭐골라야 하는지 어디 모아서 정리해둔 것 없을까요\r\n물론 ...,"[직업, 챌린저스, 서버, 시드, 컨4리, 고르, 모으, 정리, 한쪽, 주변, 처음..."
2,개초보뉴비 챌린저하고싶습니다 길을알려주세요,5471,지금 길라잡이 보스컨텐츠 순서 기준\r\n실제로는 노말루시드 / 이지 윌 잡았고\r...,"[초보, 뉴비, 챌린저, 알리, 길라잡이, 보스, 컨텐츠, 순서, 기준, 노말루시드..."
3,뉴비 레테 vs 렌 vs 보마,2400,안녕하세요\r\n아무것도 모르는 뉴비입니다 이제 메이플 깔고 캐릭 만들어야해요\r\...,"[뉴비, 레테, 보마, 아무것, 캐릭, 만들, 기본, 추천, 시작, 게시판, 공략,..."
4,보마vs레테,2120,복귀유저인데 레테랑 보마 둘중에서 고민중인데 보스는 유챔까지 생각중이고 사냥을더 선...,"[보마, 복귀, 유저, 레테랑, 보스, 사냥, 선호]"
5,뉴비 레테 바뀐체급 어떤가용? 직업정하는데 섀도어 vs레테 그리고 팔라딘..,1786,안녕하세요 읽어주셔서 감사합니다\r\n저번 3월에 자석펫 출첵 시작할때쯤 챌섭으로 ...,"[뉴비, 레테, 바뀌, 체급, 직업, 정하, 섀도어, 라딘, 감사, 저번, 자석, ..."
6,뉴비 레테 렌?,1771,손도 똥손에 노말4인 윌밖에 안깨봤는데 렌이나을까요 레테가 나을까요,"[뉴비, 레테]"
7,아란 vs 레테,1364,복귀해서 본캐 키울려고 하는데 렌은 저번 챌섭때 키웠고 보마는 별로 안땡겨서 2직업...,"[아란, 복귀, 저번, 챌섭, 보마, 땡기, 직업]"
8,메린이 보마 너프 질문,1339,이번에 복귀한 메인이인데 보우 마스터 키우려고 합니다\r\n혹시 화살 충격이랑 일점...,"[메린, 보마, 너프, 복귀, 메인이, 보우, 마스터, 화살, 충격, 폭발]"
9,본캐 추천 받습니다 와헌 vs 보마 vs 레테,1296,복귀 뉴비입니다 렌으로 진힐라까지만 잡아 봤습니다\r\n레테 개선됬다고 해서 다시 ...,"[추천, 보마, 레테, 복귀, 뉴비, 진힐라, 개선]"




[퀘스트] - 단어 '{'뉴비', '초보', '메린이', '메린', '복귀', '복귀유저', '입문', '유입'}' 포함 상위 10개 게시글


,title,views,content,tokens
0,제네시스패스 너무 어려워요,4249,이번에 챌섭으로 유입된 뉴비인데 게임이 너무 어렵습니다.\r\n스우를 잡으라는데 잡...,"[제네시스패스, 어렵, 챌섭, 유입, 뉴비, 게임, 스우, 2페이즈, 공략, 영상,..."
1,하드메이린.... 왜 안될까요,3200,배율도 어느정도 되는거 같은데 메린이라 손 문제일까요 에반이나 호영 카드가 잘 안뜨...,"[하드메이린, 메린, 문제, 에반, 호영, 카드, 미치, 도움]"
2,검밑솔...,1835,이번에 유입된 메린이라\r\n처음부터 너무 쫄아서 보스를 천천히 했거든요\r\n현재...,"[유입, 메린, 처음, 보스, 현재, 진힐라, 생존, 집중, 배율, 수준, 기준]"
3,아이템버닝 연습모드,1766,이번 시즌 쯤에 처음 시작한 뉴비인데\r\n아이템 버닝이나 챌린저스 미션? 그거 연...,"[아이템버닝, 연습, 모드, 시즌, 처음, 시작, 뉴비, 아이템, 버닝, 챌린저스,..."
4,해방하고 싶은 뉴빈데 이거 망한거임???,951,12개 제한 있는거 잘 모르고 루타비스 가엔슬 메그너스 찐모로 돌았는데 이거 망한거...,"[해방, 뉴비, 망하, 제한, 비스, 메그너스]"
5,아버 날려버렸으면 챌섭 하는 이유 크게 없겠죠?,914,올만에 복귀했는데 260까지 키운 직업이 취향이 너무 안맞아서 다시 만드려니까\r\...,"[아버, 날리, 챌섭, 이유, 복귀, 직업, 취향, 만드, 아이템, 버닝, 캐릭, ..."
6,뉴비 제발 살려주세요,862,아카이럼 선행 퀘 하는데 타밀한테 말 걸고 드래곤으로 변신해서 떨어졌는데 이상한 곳...,"[뉴비, 살리, 아카이럼, 선행, 타밀, 드래곤, 변신, 떨어지, 텔포, 방법, 퀘..."
7,뉴비 해방패스,854,강원기 디렉터 막바지에 접고\r\n이번 첼섭으로 복귀한 뉴비인데...\r\n어제 시...,"[뉴비, 해방패스, 강원기, 디렉터, 막바지, 첼섭, 복귀, 어제, 시작, 레벨, ..."
8,제네시스 해방 퀘스트 관련 질문입니다 [메린이],824,혹시 이번주\r\n주간처치 보스를 12마리 제한을 채워서\r\n매그너스를 제네시스 ...,"[제네시스, 해방, 퀘스트, 관련, 메린, 주간, 처치, 보스, 마리, 제한, 채우..."
9,제네시스 패스 질문,820,이번에 유입된 뉴비인데\r\n제네시스 패스 구매했고 지금 레벨업 중인데\r\n일단 ...,"[제네시스, 패스, 유입, 뉴비, 레벨, 화요일, 보스, 괜찮, 급하]"


In [58]:
# ============================================================
# 1. 신규·복귀 여부 컬럼 추가
# ============================================================

qna_anal["is_new_returning"] = condition.astype(bool)


# ============================================================
# 2. 분석용 전체 데이터 저장
# ============================================================

# PKL
# tokens 리스트 자료형이 그대로 보존되기 때문에
# 네트워크 분석에서는 이 파일 사용을 권장
qna_anal.to_pickle(
    OUTPUT_DIR / "qna_anal.pkl"
)


# CSV
# 사람이 확인하거나 일반적인 표 분석에 사용
qna_anal.to_csv(
    OUTPUT_DIR / "qna_anal.csv",
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 3. 신규·복귀 필터 기준 저장
# ============================================================

# 어떤 단어를 신규·복귀 기준으로 사용했는지 기록
with open(
    OUTPUT_DIR / "findwords_domain.pkl",
    "wb"
) as f:
    pickle.dump(findwords_domain, f)


# ============================================================
# 4. 저장 결과 확인
# ============================================================

print("\n===== OUTPUT 저장 완료 =====")

for file_path in sorted(OUTPUT_DIR.iterdir()):
    print(file_path.name)


===== OUTPUT 저장 완료 =====
findwords_domain.pkl
qna_anal.csv
qna_anal.pkl
